In [31]:
from dotenv import load_dotenv
from agents import Agent, Runner, OpenAIChatCompletionsModel, set_tracing_disabled, function_tool
from openai import AsyncOpenAI, OpenAI
import gradio as gr
import os
import json
import logging
from typing import Any


load_dotenv()
set_tracing_disabled(True)

In [32]:
api_key = os.getenv("NVIDIA_API_KEY")
url = os.getenv("URL")
print("url:", url)

url: https://integrate.api.nvidia.com/v1


In [33]:

MODEL_NAME="moonshotai/kimi-k2-instruct"

In [34]:
# client = OpenAI(
#   base_url = url,
#   api_key = api_key
# )


agent_client = AsyncOpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = api_key
)

In [ ]:
# async def write_n_run_python_code(message):
    
#     @function_tool
#     def run_code(code: str) -> str:
#         return "def add(a,b):\n   return a+b  output : 10"
    
#     Coder_agent = Agent(
#         name="Professional Coder Agent",
#         instructions="You write python code to answer the user's question. You can use the history of the conversation as context. You should only write code that is relevant to the user's question and the conversation history.",
#         tools=[run_code],
#         model=OpenAIChatCompletionsModel(model=MODEL_NAME, openai_client=agent_client)
#         )
#     result = await Runner.run(Coder_agent, message)

#     return result

In [ ]:
# logger = logging.getLogger(__name__)

# write_n_run_python_json = {
#     "name": "write_n_run_python_code",
#     "description": (
#         "Write and execute python code to answer the user's question. "
#         "Only use this tool if the user explicitly asks you to write python code "
#         "or if you need to write python code to answer the user's question."
#     ),
#     "parameters": {
#         "type": "object",
#         "properties": {
#             "code": {
#                 "type": "string",
#                 "description": "The python code to run. Include all necessary imports.",
#             }
#         },
#         "required": ["code"],
#     },
# }

# # Registry pattern — add new tools here instead of growing an if/elif chain
# TOOL_HANDLERS: dict[str, callable] = {
#     "write_n_run_python_code": lambda args: write_n_run_python_code(args["code"]),
# }

# tools = [{"type": "function", "function": write_n_run_python_json}]


# async def handle_tool_calls(tool_calls) -> list[dict[str, Any]]:
#     results = []
#     for tool_call in tool_calls:
#         tool_name = tool_call.function.name
#         logger.debug("Tool called: %s", tool_name)

#         try:
#             arguments = json.loads(tool_call.function.arguments)
#         except json.JSONDecodeError as e:
#             logger.error("Failed to parse arguments for %s: %s", tool_name, e)
#             result = {"error": f"Invalid arguments JSON: {e}"}
#         else:
#             handler = TOOL_HANDLERS.get(tool_name)
#             if handler is None:
#                 logger.warning("Unknown tool: %s", tool_name)
#                 result = {"error": f"Unknown tool: {tool_name!r}"}
#             else:
#                 try:
#                     result = await handler(arguments)
#                 except Exception as e:
#                     logger.exception("Tool %s raised an error", tool_name)
#                     result = {"error": str(e)}

#         results.append({
#             "role": "tool",
#             "content": json.dumps(result),
#             "tool_call_id": tool_call.id,
#         })

#     return results

In [ ]:
@function_tool
def summariser(history: list[dict]) -> dict:
    """
    Compresses a conversation history into a dense summary string.
    Use this when the context window is getting long.
    Returns a dict with a 'summary' key containing the compressed history.
    """
    if not history:
        return {"summary": ""}

    # Measure only the actual content, not Python repr noise
    raw_text = " ".join(
        msg.get("content", "") for msg in history if isinstance(msg, dict)
    )
    char_budget = max(200, int(len(raw_text) * 0.2))  # floor of 200 chars

    summariser_system_prompt = f"""You are a conversation compression engine.

Summarise the following chat history between a user and an assistant into at most {char_budget} characters.

Strict rules:
- Preserve key facts, user intent, constraints, and important context
- Remove filler, repetition, greetings, and irrelevant details
- Keep it concise, dense, and information-rich
- Use plain text (no markdown, no bullet points)
- Do NOT add new information or explanations

Output only the summary text."""

    messages_for_summarisation = (
        [{"role": "system", "content": summariser_system_prompt}]
        + history
        + [{"role": "user", "content": "Please summarise the text above."}]
    )

    try:
        summarised_history = (
            client.chat.completions.create(
                model=MODEL_NAME,
                messages=messages_for_summarisation,
            )
            .choices[0]
            .message.content
        )
    except Exception as e:
        # Fail gracefully — return the last few messages as a fallback
        fallback = " | ".join(
            msg.get("content", "")[:100]
            for msg in history[-4:]
            if isinstance(msg, dict)
        )
        print(f"Summariser failed ({e}), using fallback.")
        return {"summary": fallback}

    print("Summarised history:", summarised_history)
    return {"summary": summarised_history}



# ── Agent ────────────────────────────────────────────────────────────────────
agent = Agent(
    name="ChatAgent",
    instructions=(
        "You are a helpful assistant. "
        "When the conversation history grows long, call the summariser tool "
        "to compress it before continuing."
    ),
    model=OpenAIChatCompletionsModel(
        model=MODEL_NAME,
        openai_client=agent_client,
    ),
    tools=[summariser],
)


async def main():
    print("Chat agent ready (Ctrl+C to quit)\n")
    history = []

    while True:
        try:
            user_input = input("You: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\nBye!")
            break

        if not user_input:
            continue

        history.append({"role": "user", "content": user_input})

        result = await Runner.run(
            agent,
            input=history,
        )

        reply = result.final_output
        print(f"Agent: {reply}\n")
        history.append({"role": "assistant", "content": reply})


if __name__ == "__main__":
    import asyncio

    asyncio.run(main())



# gr.ChatInterface(chat).launch()

UserError: additionalProperties should not be set for object types. This could be because you're using an older version of Pydantic, or because you configured additional properties to be allowed. If you really need this, update the function or output tool to not use a strict schema.

In [26]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Summarised history: Summarise chat history into ≤0 characters, keeping key facts, intent, constraints; remove filler.
Summarised history: Chat empty; no content to summarise
Summarised history: Apple: round, edible fruit (red/green/yellow skin, crisp flesh, core with seeds) from Malus domestica tree; eaten fresh or cooked; source of fiber, vitamin C, antioxidants; linked to "apple a day" phrase.


Traceback (most recent call last):
  File "/home/tankaizokuo/Code/ChatBot/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 856, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tankaizokuo/Code/ChatBot/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 358, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tankaizokuo/Code/ChatBot/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2179, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tankaizokuo/Code/ChatBot/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1634, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tankaizokuo/Code/ChatBot/.venv/lib/python3.12/site-packages/gradio/utils.py", line 1027, in as

In [ ]:
# @function_tool
# def run_code(code: str) -> str:
#     """Executes the provided Python code and returns the output or error message."""
#     try:
#         # Create a local namespace for code execution
#         print("Tool is called")

#         local_namespace = {}
        
#         exec(code, {}, local_namespace)
        
#         return str(local_namespace.get("result", "Code executed successfully, but no result variable found."))
    
#     except Exception as e:
#         return f"Error executing code: {e}"


# Coder_agent = Agent(
#         name="Professional Coder Agent",
#         instructions="You write python code to answer the user's question. You can use the history of the conversation as context. You should only write code that is relevant to the user's question and the conversation history.",
#         tools=[run_code],
#         model=OpenAIChatCompletionsModel(model=MODEL_NAME, openai_client=agent_client)
# )

# tool1 = Coder_agent.as_tool(tool_name="coder_agent", tool_description="Write and execute python code to answer the user's question")




In [ ]:
result = await Runner.run(Coder_agent, message)
print(result.final_output)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
